<!-- paired-role-banner -->
> 🟧 **실습 코드 · 왼쪽 창** — `TODO`/빈 구현을 직접 완성하세요. 정답을 보기 전에 입력·출력 shape를 먼저 예측하세요.

# 22. Hybrid RAG 평가와 회귀 테스트

`data/practice/rag_queries.jsonl`을 고정 평가셋으로 사용해 sparse+dense
검색, reciprocal rank fusion(RRF),
Recall@k/MRR, no-answer threshold, 증분 index/version metadata, 회귀 리포트를 외부 API 없이 구현합니다.
17·18번을 끝낸 뒤 “좋아 보이는 검색”을 **재현 가능한 수치와 배포 gate**로 바꾸는 실습입니다.

In [ ]:
# 이 컴퓨터를 감지해 안전한 시작값을 확인합니다. 각 실습은 더 작은 값으로도 동작합니다.
from llm_engineering_lab.hardware import HardwareProfile, recommend_training_profile

LAB_HARDWARE = HardwareProfile.detect()
LAB_PROFILE = recommend_training_profile(LAB_HARDWARE)
print("hardware:", LAB_HARDWARE)
print("recommended profile:", LAB_PROFILE)

## 두 창 학습법과 목표

왼쪽 exercise의 TODO를 먼저 구현하고 오른쪽 solution으로 확인하세요.
같은 query라도 sparse/dense/hybrid의 순위가 왜 다른지, answerable과
unanswerable에서 threshold가 어떤 trade-off를 만드는지 기록합니다.
마지막에는 baseline 대비 regression을 자동 판정합니다.

In [ ]:
import hashlib, json, math, re, tempfile
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer


def find_root() -> Path:
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "data" / "practice" / "rag_queries.jsonl").exists():
            return candidate
    raise RuntimeError("코딩연습 폴더 또는 하위에서 실행하세요.")


ROOT = find_root()
KB_PATH = ROOT / "data" / "raw" / "knowledge_base.jsonl"
EVAL_PATH = ROOT / "data" / "practice" / "rag_queries.jsonl"
print(KB_PATH, EVAL_PATH, sep="\n")

## 1) corpus와 평가셋 계약

평가 query의 `relevant_ids=[]`는 no-answer 사례입니다. 정답 id가 corpus에 없거나 query id가 중복되면 지표가 거짓말하므로
평가 전에 막습니다. 평가 파일은 모델 튜닝용 train set이 아니라 고정 regression set처럼 다룹니다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=json.loads -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 1개 · 수식 0개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>json.loads(s, ...)</code></summary>

#### `json.loads(s, ...)`

- **역할:** JSON 문자열을 Python 객체로 역직렬화합니다.
- **입력·반환:** 문자열·bytes를 받고 dict·list 등의 객체를 반환합니다.
- **이 셀에서 쓰는 이유:** 메모리에 있는 JSON line이나 응답을 구조화합니다.
- **주의점:** 입력이 한 문서인지 JSON Lines의 한 줄인지 구분해야 합니다.
- **공식 문서:** [json.loads](https://docs.python.org/3/library/json.html#json.loads)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
def read_jsonl(path: Path) -> list[dict]:
    # TODO: utf-8-sig JSONL을 읽어 list로 반환하세요.
    ...


documents = read_jsonl(KB_PATH)
eval_cases = read_jsonl(EVAL_PATH)
# TODO: document/query id 고유성과 relevant_ids subset을 검증하세요.
...

<!-- api-explanations:start -->
<!-- api-explanations:keys=pd.DataFrame -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 1개 · 수식 0개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>pd.DataFrame(data=None, index=None, columns=None, ...)</code></summary>

#### `pd.DataFrame(data=None, index=None, columns=None, ...)`

- **역할:** 행·열 구조의 DataFrame을 만듭니다.
- **입력·반환:** 표 데이터와 열·그룹 설정을 받고 DataFrame, Series 또는 집계 객체를 반환합니다.
- **이 셀에서 쓰는 이유:** 학습 전 데이터 분포·결측·중복과 분할 기준을 점검합니다.
- **주의점:** dtype 추론, 결측값, index 보존과 grouping 후 shape 변화를 확인해야 합니다.
- **공식 문서:** [pd.DataFrame](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
# TODO: 평가셋의 query 길이, relevant 수, answerable 여부를 DataFrame으로 확인하세요.
eval_frame = ...
display(eval_frame)

## 2) sparse index와 version metadata

문자 n-gram TF-IDF를 sparse 기준선으로 만듭니다. index version에는
corpus hash, document 수, embedding 설정을 기록합니다.
단순 날짜보다 content hash가 같은 데이터/설정에서 같은 artifact임을 확인하기 쉽습니다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=json.dumps,dataclass -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 2개 · 수식 0개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>json.dumps(obj, ensure_ascii=True, indent=None, ...)</code></summary>

#### `json.dumps(obj, ensure_ascii=True, indent=None, ...)`

- **역할:** Python 객체를 JSON 문자열로 직렬화합니다.
- **입력·반환:** 객체를 받고 JSON 문자열을 반환합니다.
- **이 셀에서 쓰는 이유:** metric이나 sample을 파일·로그에 넣기 전 안정된 문자열로 만듭니다.
- **주의점:** 한글 보존에는 `ensure_ascii=False`가 필요하고 객체 종류가 제한됩니다.
- **공식 문서:** [json.dumps](https://docs.python.org/3/library/json.html#json.dumps)

</details>

<details>
<summary><code>@dataclass(frozen=False, slots=False, ...)</code></summary>

#### `@dataclass(frozen=False, slots=False, ...)`

- **역할:** type annotation을 바탕으로 초기화·표현 메서드 등을 생성합니다.
- **입력·반환:** class decorator 설정을 받고 변환된 class를 반환합니다.
- **이 셀에서 쓰는 이유:** 실험 설정과 결과 record의 필드를 명시적으로 정의합니다.
- **주의점:** mutable 기본값은 `default_factory`를 쓰고 equality·frozen 의미를 확인합니다.
- **공식 문서:** [dataclass](https://docs.python.org/3/library/dataclasses.html#dataclasses.dataclass)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
def document_text(row: dict) -> str:
    return " ".join((row["title"], row["category"], *row["keywords"], row["content"]))


def corpus_hash(records: list[dict]) -> str:
    # TODO: id 정렬 후 canonical JSON의 SHA-256 앞 12자를 반환하세요.
    ...


@dataclass
class RankedHit:
    document_id: str
    rank: int
    score: float
    retriever: str

<!-- api-explanations:start -->
<!-- api-explanations:keys=np.argsort -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 1개 · 수식 0개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>np.argsort(a, axis=-1, kind=None)</code></summary>

#### `np.argsort(a, axis=-1, kind=None)`

- **역할:** 정렬 결과를 만드는 index 순서를 반환합니다.
- **입력·반환:** 배열과 축·dtype 등의 설정을 받고 배열 또는 수치 결과를 반환합니다.
- **이 셀에서 쓰는 이유:** 데이터 shape과 수치 연산을 논문 실험에 맞게 구성·검증합니다.
- **주의점:** 축, dtype, 복사·view 여부와 빈 배열에서의 동작을 확인해야 합니다.
- **공식 문서:** [np.argsort](https://numpy.org/doc/stable/reference/generated/numpy.argsort.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
class SparseIndex:
    def fit(self, records: list[dict]):
        """TODO: char_wb TF-IDF와 version metadata를 만드세요."""
        ...

    def search(self, query: str, top_k: int = 5) -> list[RankedHit]:
        """TODO: normalized TF-IDF 내적으로 top-k를 반환하세요."""
        ...

In [ ]:
sparse_index = SparseIndex().fit(documents)
# TODO: version과 401 query 검색 결과를 확인하세요.
...

## 3) deterministic dense embedding: TF-IDF + SVD

외부 모델 다운로드 없이 dense vector를 만들기 위해 TF-IDF를 TruncatedSVD로 투영합니다. 이는 의미 학습 모델이 아니라
latent lexical baseline이지만 sparse/dense 결합, 정규화, dimension, index 계약을 재현 가능하게 연습할 수 있습니다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=np.linalg.norm -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 1개 · 수식 1개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>np.linalg.norm(x, ord=None, axis=None, ...)</code></summary>

#### `np.linalg.norm(x, ord=None, axis=None, ...)`

- **역할:** 벡터나 행렬의 norm을 계산합니다.
- **입력·반환:** 배열과 축·dtype 등의 설정을 받고 배열 또는 수치 결과를 반환합니다.
- **이 셀에서 쓰는 이유:** 데이터 shape과 수치 연산을 논문 실험에 맞게 구성·검증합니다.
- **주의점:** 축, dtype, 복사·view 여부와 빈 배열에서의 동작을 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
\lVert x\rVert_p=\left(\sum_i\lvert x_i\rvert^p\right)^{1/p},\qquad \lVert x\rVert_2=\sqrt{\sum_i x_i^2}
$$

- **기호:** $p$는 norm 차수이며 $p=2$가 유클리드 길이입니다.
- **수식 → 코드:** `ord`와 `dim`·`axis`가 어떤 원소 묶음의 크기를 계산할지 정합니다.
- **직관:** 여러 성분을 하나의 거리·크기 값으로 요약합니다.
- **shape 확인:** 지정한 축은 결과에서 제거되며 `keepdim=True`이면 크기 1로 남습니다.
- **공식 문서:** [np.linalg.norm](https://numpy.org/doc/stable/reference/generated/numpy.linalg.norm.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
class DenseSVDIndex:
    def fit(self, records: list[dict], n_components: int = 10):
        """TODO: TF-IDF→SVD→L2 normalize dense matrix와 version을 만드세요."""
        ...

    def search(self, query: str, top_k: int = 5) -> list[RankedHit]:
        """TODO: 같은 transform/normalize 후 cosine top-k를 반환하세요."""
        ...

<!-- api-explanations:start -->
<!-- api-explanations:keys=np.allclose -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 1개 · 수식 1개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>np.allclose(a, b, rtol=1e-5, atol=1e-8, equal_nan=False)</code></summary>

#### `np.allclose(a, b, rtol=1e-5, atol=1e-8, equal_nan=False)`

- **역할:** 두 배열의 모든 원소가 허용 오차 안에서 가까운지 검사합니다.
- **입력·반환:** 배열 `a`, 기준 배열 `b`, 오차 설정을 받고 Python `bool` 하나를 반환합니다.
- **이 셀에서 쓰는 이유:** 부동소수점 계산 결과를 정확한 값이 아니라 수치적으로 같은지 확인합니다.
- **주의점:** 조건은 `abs(a-b) <= atol + rtol*abs(b)`입니다. `==`와 달리 오차를 허용하고, `b`가 기준이라 비대칭일 수 있으며 1보다 매우 작은 값에는 기본 `atol`이 너무 클 수 있습니다.

##### 관련 수식과 코드 연결

$$
\lvert a_i-b_i\rvert \le \mathrm{atol}+\mathrm{rtol}\,\lvert b_i\rvert
$$

- **기호:** $a_i$는 계산값, $b_i$는 기준값, `atol`은 절대 오차, `rtol`은 상대 오차 허용치입니다.
- **수식 → 코드:** `isclose`는 이 부등식을 원소마다 계산하고, `allclose`는 모든 결과에 논리 AND를 적용합니다.
- **직관:** 완전히 같은 비트인지가 아니라 실험 목적에 충분히 가까운지를 묻는 검사입니다.
- **shape 확인:** 두 입력은 broadcasting 가능해야 합니다. `isclose`는 broadcast shape, `allclose`는 scalar `bool`을 반환합니다.
- **공식 문서:** [np.allclose](https://numpy.org/doc/stable/reference/generated/numpy.allclose.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
dense_index = DenseSVDIndex().fit(documents, n_components=10)
# TODO: vector shape/norm/version을 검증하고 sparse top-3와 비교하세요.
...

## 4) Reciprocal Rank Fusion

sparse score와 dense score는 척도가 다르므로 직접 더하지 않습니다. RRF는 각 ranking의 `1/(rrf_k + rank)`를 더해
score calibration 없이 순위를 결합합니다. 한 retriever에만 등장한 문서도 포함됩니다.

In [ ]:
def reciprocal_rank_fusion(
    rankings: list[list[RankedHit]], top_k: int = 5, rrf_k: int = 60
) -> list[RankedHit]:
    """TODO: document_id별 reciprocal rank를 합쳐 hybrid 결과를 반환하세요."""
    ...


def hybrid_search(query: str, top_k: int = 5) -> list[RankedHit]:
    # TODO: 후보 폭을 늘려 sparse/dense 결과를 RRF로 결합하세요.
    ...

In [ ]:
fusion_query = "운송장 조회가 이틀째 그대로예요"
# TODO: sparse/dense/hybrid ranking을 한 표로 비교하세요.
fusion_frame = ...
display(fusion_frame)

## 5) Recall@k와 MRR

여러 relevant id가 있으면 하나라도 top-k에 들어오면 Recall hit로 보고,
첫 relevant의 reciprocal rank를 MRR에 사용합니다.
no-answer case는 retrieval ranking metric에서 제외하고 threshold 평가에서 별도로 다룹니다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=np.mean -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 1개 · 수식 1개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>np.mean(a, axis=None, dtype=None, ...)</code></summary>

#### `np.mean(a, axis=None, dtype=None, ...)`

- **역할:** 지정 축의 산술평균을 계산합니다.
- **입력·반환:** 배열과 축·dtype 등의 설정을 받고 배열 또는 수치 결과를 반환합니다.
- **이 셀에서 쓰는 이유:** 데이터 shape과 수치 연산을 논문 실험에 맞게 구성·검증합니다.
- **주의점:** 축, dtype, 복사·view 여부와 빈 배열에서의 동작을 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
\bar x=\frac{1}{N}\sum_{i=1}^{N}x_i
$$

- **기호:** $N$은 선택한 축의 원소 수입니다.
- **수식 → 코드:** `dim` 또는 `axis`가 평균에 포함할 원소와 결과에서 사라질 축을 결정합니다.
- **직관:** 여러 관측값을 중심 위치 하나로 요약합니다.
- **shape 확인:** `keepdim=False`이면 축약한 축이 제거되고 참이면 크기 1로 남습니다.
- **공식 문서:** [np.mean](https://numpy.org/doc/stable/reference/generated/numpy.mean.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
def ranking_metrics(search_fn, cases: list[dict], k: int) -> dict:
    """TODO: answerable case의 recall@k와 MRR을 계산하세요."""
    ...

In [ ]:
searchers = {
    "sparse": sparse_index.search,
    "dense_svd": dense_index.search,
    "hybrid_rrf": hybrid_search,
}
# TODO: 각 retriever의 k=1,3,5 지표를 비교하세요.
ranking_report = ...
display(ranking_report)

## 6) no-answer threshold

RRF score는 작고 좁은 범위라 no-answer threshold 설명에 불편합니다.
여기서는 hybrid top-1 문서에 대해 sparse/dense cosine의
최댓값을 confidence proxy로 사용합니다. 이 값도 확률은 아니며 validation distribution에서 threshold를 정해야 합니다.

In [ ]:
def hybrid_confidence(query: str, result: list[RankedHit]) -> float:
    """TODO: hybrid top-1 id의 sparse/dense score 중 최댓값을 반환하세요."""
    ...


def threshold_metrics(threshold: float) -> dict:
    """TODO: coverage, answerable accuracy, no-answer accuracy를 계산하세요."""
    ...

In [ ]:
# TODO: threshold 0.00~0.30의 trade-off 표를 만들고 정책 threshold를 선택하세요.
threshold_report = ...
POLICY_THRESHOLD = ...
display(threshold_report)

## 7) 증분 update와 version metadata

TF-IDF/SVD는 vocabulary와 global statistics를 학습하므로 진짜
append-only update가 아닙니다. 작은 corpus에서는 새 문서를 합친 뒤
재학습하는 것이 안전합니다. 이를 `update()`라는 API로 감싸되
parent hash, 새 hash, generation을 기록해 lineage를 보존합니다.

In [ ]:
class VersionedHybridIndex:
    def __init__(
        self, records: list[dict], generation: int = 1, parent_hash: str | None = None
    ):
        # TODO: ids 고유성 검증, sparse/dense fit, version metadata를 만드세요.
        ...

    def search(self, query: str, top_k: int = 5) -> list[RankedHit]:
        # TODO: 현재 두 index의 RRF 검색을 구현하세요.
        ...

    def update(self, new_records: list[dict]):
        """TODO: 중복 id를 막고 merged corpus로 다음 generation을 반환하세요."""
        ...

In [ ]:
versioned_v1 = VersionedHybridIndex(documents)
new_document = {
    "id": "kb-013",
    "title": "회사 주소 안내",
    "category": "company",
    "content": "본사 주소는 서울시 가상구 학습로 22이며 방문은 사전 예약이 필요합니다.",
    "keywords": ["회사 주소", "본사", "방문"],
}
# TODO: update 후 generation/parent hash/document count와 새 문서 검색을 검증하세요.
versioned_v2 = ...

## 8) regression evaluation report

배포 전 후보 index를 고정 평가셋에 돌려 baseline과 비교합니다.
corpus update로 q-013이 이제 answerable해졌으므로 평가 label도
version을 올려 갱신해야 합니다. 여기서는 기존 12개 answerable query에 대한 Recall/MRR 하락을 먼저 gate로 사용합니다.

In [ ]:
def regression_metrics(
    index: VersionedHybridIndex, cases: list[dict], k: int = 3
) -> dict:
    """TODO: answerable cases의 recall@k/MRR 및 latency 대신 query count/version을 기록하세요."""
    ...


baseline_metrics = regression_metrics(versioned_v1, eval_cases)
candidate_metrics = regression_metrics(versioned_v2, eval_cases)

In [ ]:
def build_regression_report(
    baseline: dict, candidate: dict, tolerance: float = 0.02
) -> dict:
    """TODO: metric delta와 pass/fail, version lineage를 포함하세요."""
    ...


regression_report = build_regression_report(baseline_metrics, candidate_metrics)
print(json.dumps(regression_report, ensure_ascii=False, indent=2))

## 9) query별 회귀 분석

집계 지표가 같아도 서로 다른 query가 좋아지고 나빠질 수 있습니다. baseline/candidate에서 정답 첫 순위를 비교하고,
특히 순위가 하락한 query와 no-answer가 새 문서 추가로 answerable이 된 query를 사람이 검토합니다.

In [ ]:
def first_relevant_rank(index, case: dict, k: int = 5):
    # TODO: 첫 relevant rank 또는 None을 반환하세요.
    ...


# TODO: query별 baseline/candidate rank와 변화량을 표로 만드세요.
per_query_report = ...
display(per_query_report)

## 완료 기준과 확장

- sparse/dense의 score를 직접 섞지 않고 RRF가 필요한 이유를 설명할 수 있다.
- Recall@k/MRR과 no-answer 정확도를 분리하고 threshold의 coverage trade-off를 읽을 수 있다.
- corpus hash/generation/parent hash로 index lineage를 추적할 수 있다.
- 평균 지표뿐 아니라 query별 rank regression을 배포 gate에 포함할 수 있다.

다음 실험: dense SVD dimension, RRF `k`, candidate 폭, threshold를 바꾸되 **평가셋을 보며 규칙을 계속 추가하지 말고**
별도 validation/test split을 만드세요. 실제 embedding/FAISS로 교체해도 이 평가 harness는 그대로 유지할 수 있습니다.